In [1]:
from pathlib import Path
import duckdb

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_DIR = PROJECT_ROOT / "outputs" / "part2" / "raw"
PART3_DIR = PROJECT_ROOT / "outputs" / "part3"
PART3_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = PART3_DIR / "firmable.duckdb"

con = duckdb.connect(str(DB_PATH))

print("DuckDB:", DB_PATH)
print("Raw data:", RAW_DIR)

DuckDB: /Users/ario/Documents/firmable/outputs/part3/firmable.duckdb
Raw data: /Users/ario/Documents/firmable/outputs/part2/raw


In [2]:
source_files = sorted([
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {
        ".csv", ".tsv", ".xlsx", ".xls", ".json", ".jsonl", ".xml"
    }
])

for i, p in enumerate(source_files, 1):
    print(i, p.relative_to(RAW_DIR))

1 1ca29b03-220d-4c19-b34e-c7932cd4508d/e70e6e0b-1a62-456b-a3d1-cd8f7508b54f/Victorian_Government_Schools_ABNs.xlsx
2 4117fdec-3eff-44a1-9b78-869f7ec5c409/0f2e41ec-6f49-4d35-8e6d-ebcd4a0e3e27/Registered_Auditor_Dataset_Current.csv
3 5bd7fcab-e315-42cb-8daf-50b7efc2027e/0ae4d427-6fa8-4d40-8e76-c6909b5a071b/extracted/20260923_Public01.xml
4 6f1cc305-a8b2-4748-bd40-c00754ad6504/34f8f3b8-68b2-41b5-ac3f-9ff01da70c5c/Jan_2023.xlsx
5 732d6bf3-2670-4226-9235-789bcf670288/b8575bd0-16d4-452e-a4ec-db48330e6356/Business_establishments_location_and_industry_classification_CSV.csv
6 7b8656f9-606d-4337-af29-66b89b2eeefb/5c3914e6-413e-4a2c-b890-bf8efe3eabf2/Company_Dataset_Current.csv
7 ab7eddce-84df-4098-bc8f-500d0d9776d1/d98a113d-6b50-40e6-b65f-2612efc877f4/AFS_Licensee_Dataset_Current.csv
8 b050b242-4487-4306-abf5-07ca073e5594/8fb32972-24e9-4c95-885e-7140be51be8a/ACNC_Register_of_Australian_charities_CSV.csv


In [3]:
import pandas as pd

sources = []

for p in source_files:
    rel = p.relative_to(RAW_DIR)
    parts = rel.parts

    dataset_id = parts[0]
    resource_id = parts[1]

    name = p.stem.lower()
    name = "".join(c if c.isalnum() else "_" for c in name)
    name = "_".join(filter(None, name.split("_")))

    sources.append({
        "dataset_id": dataset_id,
        "resource_id": resource_id,
        "table_name": f"src_{name}",
        "format": p.suffix.lower().lstrip("."),
        "path": str(p),
    })

sources_df = pd.DataFrame(sources)

sources_df

,dataset_id,resource_id,table_name,format,path
0,1ca29b03-220d-4c19-b34e-c7932cd4508d,e70e6e0b-1a62-456b-a3d1-cd8f7508b54f,src_victorian_government_schools_abns,xlsx,/Users/ario/Documents/firmable/outputs/part2/r...
1,4117fdec-3eff-44a1-9b78-869f7ec5c409,0f2e41ec-6f49-4d35-8e6d-ebcd4a0e3e27,src_registered_auditor_dataset_current,csv,/Users/ario/Documents/firmable/outputs/part2/r...
2,5bd7fcab-e315-42cb-8daf-50b7efc2027e,0ae4d427-6fa8-4d40-8e76-c6909b5a071b,src_20260923_public01,xml,/Users/ario/Documents/firmable/outputs/part2/r...
3,6f1cc305-a8b2-4748-bd40-c00754ad6504,34f8f3b8-68b2-41b5-ac3f-9ff01da70c5c,src_jan_2023,xlsx,/Users/ario/Documents/firmable/outputs/part2/r...
4,732d6bf3-2670-4226-9235-789bcf670288,b8575bd0-16d4-452e-a4ec-db48330e6356,src_business_establishments_location_and_indus...,csv,/Users/ario/Documents/firmable/outputs/part2/r...
5,7b8656f9-606d-4337-af29-66b89b2eeefb,5c3914e6-413e-4a2c-b890-bf8efe3eabf2,src_company_dataset_current,csv,/Users/ario/Documents/firmable/outputs/part2/r...
6,ab7eddce-84df-4098-bc8f-500d0d9776d1,d98a113d-6b50-40e6-b65f-2612efc877f4,src_afs_licensee_dataset_current,csv,/Users/ario/Documents/firmable/outputs/part2/r...
7,b050b242-4487-4306-abf5-07ca073e5594,8fb32972-24e9-4c95-885e-7140be51be8a,src_acnc_register_of_australian_charities_csv,csv,/Users/ario/Documents/firmable/outputs/part2/r...


In [4]:
con.register("sources_df", sources_df)

con.execute("""
CREATE OR REPLACE TABLE source_registry AS
SELECT *
FROM sources_df
""")

con.sql("""
SELECT
    dataset_id,
    resource_id,
    table_name,
    format
FROM source_registry
""").show()

┌──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────────────────────────────────────┬─────────┐
│              dataset_id              │             resource_id              │                              table_name                              │ format  │
│               varchar                │               varchar                │                               varchar                                │ varchar │
├──────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────────────────────────────────────┼─────────┤
│ 1ca29b03-220d-4c19-b34e-c7932cd4508d │ e70e6e0b-1a62-456b-a3d1-cd8f7508b54f │ src_victorian_government_schools_abns                                │ xlsx    │
│ 4117fdec-3eff-44a1-9b78-869f7ec5c409 │ 0f2e41ec-6f49-4d35-8e6d-ebcd4a0e3e27 │ src_registered_auditor_dataset_current                               │ csv     │
│ 5bd7fcab-e315-42cb-8daf-50b7efc2

In [5]:
def inspect_source(row):
    path = Path(row["path"])
    fmt = row["format"]

    print("\n", "=" * 80)
    print(row["table_name"])
    print(path.name)

    if fmt in {"csv", "tsv"}:
        df = pd.read_csv(path, nrows=5, sep=None, engine="python")

    elif fmt in {"xlsx", "xls"}:
        df = pd.read_excel(path, nrows=5)

    elif fmt == "xml":
        print("XML — inspect separately")
        return

    else:
        print("Skipped:", fmt)
        return

    print(list(df.columns))
    display(df.head(3))


for _, row in sources_df.iterrows():
    inspect_source(row)


src_victorian_government_schools_abns
Victorian_Government_Schools_ABNs.xlsx
['SchoolNumber', 'SchoolName', 'Fin_Period', 'LoadDate', 'ABN']


,SchoolNumber,SchoolName,Fin_Period,LoadDate,ABN
0,1,Alberton Primary School,201506,2015-07-14 06:30:43.000,35954747085
1,3,Allansford and District Primary School,201506,2015-07-14 06:30:43.863,37051817052
2,4,Avoca Primary School,201506,2015-07-07 06:36:29.823,39744336864



src_registered_auditor_dataset_current
Registered_Auditor_Dataset_Current.csv
['\ufeffREGISTER_NAME', 'REG_AUD_NUM', 'REG_AUD_NAME', 'REG_AUD_ACN', 'REG_AUD_START_DT', 'REG_AUD_STATUS', 'REG_AUD_SUSP_DT', 'REG_AUD_ADD_LOCAL', 'REG_AUD_ADD_STATE', 'REG_AUD_ADD_PCODE', 'REG_AUD_ADD_COUNTRY']


,﻿REGISTER_NAME,REG_AUD_NUM,REG_AUD_NAME,REG_AUD_ACN,REG_AUD_START_DT,REG_AUD_STATUS,REG_AUD_SUSP_DT,REG_AUD_ADD_LOCAL,REG_AUD_ADD_STATE,REG_AUD_ADD_PCODE,REG_AUD_ADD_COUNTRY
0,Registered Auditors,339233,A D DANIELI AUDIT PTY LTD,136616610.0,24/07/2009,APPR,NaN,SYDNEY,NSW,2000,Australia
1,Registered Auditors,486826,"ABBOTT, ALASTAIR GORDON",NaN,10/05/2016,APPR,NaN,EAST PERTH,WA,6004,Australia
2,Registered Auditors,517142,"ABBOTT, BELINDA KATE",NaN,17/07/2019,APPR,NaN,MALVERN EAST,VIC,3145,Australia



src_20260923_public01
20260923_Public01.xml
XML — inspect separately

src_jan_2023
Jan_2023.xlsx
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20']


...

,﻿Company Name,ACN,Type,Class,Sub Class,Status,Date of Registration,Date of Deregistration,Previous State of Registration,State Registration number,Modified since last report,Current Name Indicator,ABN,Current Name,Current Name Start Date
0,LOVINI HOLDINGS PTY LTD,19,APTY,LMSH,PROP,REGD,08/01/1990,NaN,NSW,46869041,NaN,NaN,89000000019,MONAKA PTY LTD,28/01/2016
1,MONAKA PTY LTD,19,APTY,LMSH,PROP,REGD,08/01/1990,NaN,NSW,46869041,NaN,Y,89000000019,NaN,NaN
2,CHRISTENSEN & ASSOCIATES PTY LTD,28,APTY,LMSH,PSTC,REGD,15/09/1987,NaN,NSW,40398926,NaN,Y,91000000028,NaN,NaN



src_afs_licensee_dataset_current
AFS_Licensee_Dataset_Current.csv
['\ufeffREGISTER_NAME', 'AFS_LIC_NUM', 'AFS_LIC_NAME', 'AFS_LIC_ABN_ACN', 'AFS_LIC_START_DT', 'AFS_LIC_PRE_FSR', 'AFS_LIC_ADD_LOCAL', 'AFS_LIC_ADD_STATE', 'AFS_LIC_ADD_PCODE', 'AFS_LIC_ADD_COUNTRY', 'AFS_LIC_CONDITION']


,﻿REGISTER_NAME,AFS_LIC_NUM,AFS_LIC_NAME,AFS_LIC_ABN_ACN,AFS_LIC_START_DT,AFS_LIC_PRE_FSR,AFS_LIC_ADD_LOCAL,AFS_LIC_ADD_STATE,AFS_LIC_ADD_PCODE,AFS_LIC_ADD_COUNTRY,AFS_LIC_CONDITION
0,AFS Licence,218600,IPIB PTY LTD,85007186003,02/05/2002,"""General insurance broker"",""000030366""",SOUTHBANK,VIC,3006,Australia,"""This licence authorises the licensee to carry..."
1,AFS Licence,218678,IMC PACIFIC PTY LTD,89099273846,26/04/2002,NaN,SYDNEY,NSW,2000,Australia,"""This licence authorises the licensee to carry..."
2,AFS Licence,218770,TRILOGY GROUP AUSTRALIA PTY LTD,80078111654,22/08/2002,"""Futures broker"",""000169201""",TOORADIN,VIC,3980,Australia,"""This licence authorises the licensee to carry..."



src_acnc_register_of_australian_charities_csv
ACNC_Register_of_Australian_charities_CSV.csv
['ABN', 'Charity_Legal_Name', 'Other_Organisation_Names', 'Address_Type', 'Address_Line_1', 'Address_Line_2', 'Address_Line_3', 'Town_City', 'State', 'Postcode', 'Country', 'Charity_Website', 'Registration_Date', 'Date_Organisation_Established', 'Charity_Size', 'Number_of_Responsible_Persons', 'Financial_Year_End', 'Operates_in_ACT', 'Operates_in_NSW', 'Operates_in_NT', 'Operates_in_QLD', 'Operates_in_SA', 'Operates_in_TAS', 'Operates_in_VIC', 'Operates_in_WA', 'Operating_Countries', 'PBI', 'HPC', 'Preventing_or_relieving_suffering_of_animals', 'Advancing_Culture', 'Advancing_Education', 'Advancing_Health', 'Promote_or_oppose_a_change_to_law__government_poll_or_prac', 'Advancing_natual_environment', 'Promoting_or_protecting_human_rights', 'Purposes_beneficial_to_ther_general_public_and_other_analogous', 'Promoting_reconciliation__mutual_respect_and_tolerance', 'Advancing_Religion', 'Advancing_s

,ABN,Charity_Legal_Name,Other_Organisation_Names,Address_Type,Address_Line_1,Address_Line_2,Address_Line_3,Town_City,State,Postcode,...,Pre_Post_Release_Offenders,Rural_Regional_Remote_Communities,Unemployed_Person,Veterans_or_their_families,Victims_of_crime,Victims_of_Disasters,Youth,animals,environment,other_gender_identities
0,20613885410,Marradhali Aboriginal Corporation,NaN,Business,73 Taylor Street,NaN,NaN,Dubbo,NSW,2830,...,Y,Y,Y,NaN,NaN,NaN,Y,NaN,NaN,NaN
1,45629238500,Margate Family Op Shop Trust,NaN,Business,32 Baynes St,NaN,NaN,Margate,QLD,4019,...,NaN,NaN,Y,NaN,Y,Y,Y,NaN,NaN,NaN
2,72886162186,Jonahs Wings,NaN,Business,44 Reigate Rd,NaN,NaN,Highton,VIC,3216,...,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN


In [6]:
for _, row in sources_df.iterrows():
    if row["format"] not in {"csv", "tsv"}:
        continue

    table = row["table_name"]
    path = row["path"].replace("'", "''")

    print("Loading:", table)

    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT
            row_number() OVER () AS row_id,
            *
        FROM read_csv_auto(
            '{path}',
            header = true,
            sample_size = 100000,
            ignore_errors = true
        )
    """)

    count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print("Rows:", f"{count:,}")

Loading: src_registered_auditor_dataset_current
Rows: 2,965
Loading: src_business_establishments_location_and_industry_classification_csv
Rows: 413,550
Loading: src_company_dataset_current
Rows: 4,448,915
Loading: src_afs_licensee_dataset_current
Rows: 6,526
Loading: src_acnc_register_of_australian_charities_csv
Rows: 66,399


In [7]:
for _, row in sources_df.iterrows():
    if row["format"] not in {"xlsx", "xls"}:
        continue

    table = row["table_name"]
    path = row["path"]

    print("Loading:", table)

    df = pd.read_excel(path)

    con.register("_excel_df", df)

    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT
            row_number() OVER () AS row_id,
            *
        FROM _excel_df
    """)

    con.unregister("_excel_df")

    count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print("Rows:", f"{count:,}")

Loading: src_victorian_government_schools_abns
Rows: 1,540
Loading: src_jan_2023
Rows: 24,200


In [8]:
from lxml import etree
import json

xml_row = sources_df[sources_df["format"] == "xml"].iloc[0]

xml_path = xml_row["path"]
table = "src_abn_bulk_extract"

con.execute(f"""
CREATE OR REPLACE TABLE {table} (
    row_id BIGINT,
    raw_json JSON
)
""")

batch = []
batch_size = 10_000
row_id = 0

for _, elem in etree.iterparse(
    xml_path,
    events=("end",),
    tag="{*}ABR"
):
    row_id += 1

    def element_to_dict(node):
        obj = {}

        for k, v in node.attrib.items():
            obj[f"@{k}"] = v

        children = list(node)

        if not children:
            text = (node.text or "").strip()
            return text if text else obj

        for child in children:
            key = etree.QName(child).localname
            value = element_to_dict(child)

            if key in obj:
                if not isinstance(obj[key], list):
                    obj[key] = [obj[key]]
                obj[key].append(value)
            else:
                obj[key] = value

        return obj

    batch.append({
        "row_id": row_id,
        "raw_json": json.dumps(element_to_dict(elem))
    })

    elem.clear()

    while elem.getprevious() is not None:
        del elem.getparent()[0]

    if len(batch) >= batch_size:
        df = pd.DataFrame(batch)
        con.register("_xml_batch", df)

        con.execute(f"""
            INSERT INTO {table}
            SELECT row_id, raw_json::JSON
            FROM _xml_batch
        """)

        con.unregister("_xml_batch")
        batch = []

if batch:
    df = pd.DataFrame(batch)
    con.register("_xml_batch", df)

    con.execute(f"""
        INSERT INTO {table}
        SELECT row_id, raw_json::JSON
        FROM _xml_batch
    """)

    con.unregister("_xml_batch")

print("Rows:", f"{row_id:,}")

Rows: 1,027,700


In [9]:
tables = con.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

tables

,table_name
0,source_registry
1,sources_df
2,src_abn_bulk_extract
3,src_acnc_register_of_australian_charities_csv
4,src_afs_licensee_dataset_current
5,src_business_establishments_location_and_indus...
6,src_company_dataset_current
7,src_jan_2023
8,src_registered_auditor_dataset_current
9,src_victorian_government_schools_abns


In [10]:
for table in tables["table_name"]:
    if table.startswith("src_"):
        count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {count:,}")

src_abn_bulk_extract: 1,027,700
src_acnc_register_of_australian_charities_csv: 66,399
src_afs_licensee_dataset_current: 6,526
src_business_establishments_location_and_industry_classification_csv: 413,550
src_company_dataset_current: 4,448,915
src_jan_2023: 24,200
src_registered_auditor_dataset_current: 2,965
src_victorian_government_schools_abns: 1,540


In [11]:
for table in tables["table_name"]:
    if not table.startswith("src_"):
        continue

    cols = con.execute(f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = '{table}'
        ORDER BY ordinal_position
    """).df()["column_name"].tolist()

    id_cols = [
        c for c in cols
        if "abn" in c.lower()
        or "acn" in c.lower()
        or "nzbn" in c.lower()
    ]

    print("\n", table)
    print(id_cols)


 src_abn_bulk_extract
[]

 src_acnc_register_of_australian_charities_csv
['ABN']

 src_afs_licensee_dataset_current
['AFS_LIC_ABN_ACN']

 src_business_establishments_location_and_industry_classification_csv
[]

 src_company_dataset_current
['ACN', 'ABN']

 src_jan_2023
[]

 src_registered_auditor_dataset_current
['REG_AUD_ACN']

 src_victorian_government_schools_abns
['ABN']


In [12]:
con.sql("""
SELECT
    row_id,
    json_keys(raw_json) AS top_level_keys,
    raw_json
FROM src_abn_bulk_extract
LIMIT 3
""").show(max_width=200)

┌────────┬──────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────┐
│ row_id │                                        top_level_keys                                        │                                           raw_json                                           │
│ int64  │                                          varchar[]                                           │                                             json                                             │
├────────┼──────────────────────────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────┤
│      1 │ [@recordLastUpdatedDate, @replaced, ABN, EntityType, MainEntity, ASICNumber, GST,            │ {"@recordLastUpdatedDate": "20180216", "@replaced": "N", "ABN": "11000000948", "EntityType

In [13]:
con.sql("""
SELECT
    row_id,
    raw_json->'ABN' AS abn_object
FROM src_abn_bulk_extract
LIMIT 10
""").show(max_width=200)

┌────────┬───────────────┐
│ row_id │  abn_object   │
│ int64  │     json      │
├────────┼───────────────┤
│      1 │ "11000000948" │
│      2 │ "11000002568" │
│      3 │ "11000003314" │
│      4 │ "11000003378" │
│      5 │ "11000007876" │
│      6 │ "11000009496" │
│      7 │ "11000013098" │
│      8 │ "11000015976" │
│      9 │ "11000016722" │
│     10 │ "11000017596" │
└────────┴───────────────┘
  10 rows      2 columns



In [14]:
con.sql("""
SELECT
    row_id,
    raw_json->>'ABN' AS abn,
    raw_json->>'ASICNumber' AS acn,
    raw_json->'MainEntity'->'NonIndividualName'->>'NonIndividualNameText' AS preferred_name
FROM src_abn_bulk_extract
LIMIT 20
""").show(max_width=120)

┌────────┬─────────────┬───────────┬──────────────────────────────────────────┐
│ row_id │     abn     │    acn    │              preferred_name              │
│ int64  │   varchar   │  varchar  │                 varchar                  │
├────────┼─────────────┼───────────┼──────────────────────────────────────────┤
│      1 │ 11000000948 │ 000000948 │ QBE INSURANCE (INTERNATIONAL) LTD        │
│      2 │ 11000002568 │ 000002568 │ TOOHEYS PTY LIMITED                      │
│      3 │ 11000003314 │ 000003314 │ NEWCASTLE GOLF CLUB LTD                  │
│      4 │ 11000003378 │ 000003378 │ BBC HARDWARE LIMITED                     │
│      5 │ 11000007876 │ 000007876 │ AMPOL PETROLEUM PTY LTD                  │
│      6 │ 11000009496 │ 000009496 │ PIONEER STEEL PTY LTD                    │
│      7 │ 11000013098 │ 000013098 │ SYDNEY NIGHT PATROL & INQUIRY CO PTY LTD │
│      8 │ 11000015976 │ 000015976 │ MIRROR NEWSPAPERS PTY LIMITED            │
│      9 │ 11000016722 │ 000016722 │ INS

In [15]:
con.execute("""
CREATE OR REPLACE TABLE firmable_entities AS

SELECT
    row_number() OVER (ORDER BY abn) AS entity_id,
    abn,
    acn,
    legal_name,
    entity_type,
    status

FROM (
    SELECT
        raw_json->>'ABN' AS abn,
        NULLIF(raw_json->>'ASICNumber', '') AS acn,

        raw_json
            ->'MainEntity'
            ->'NonIndividualName'
            ->>'NonIndividualNameText' AS legal_name,

        raw_json
            ->'EntityType'
            ->>'EntityTypeText' AS entity_type,

        CASE
            WHEN raw_json->'ABN'->>'@status' = 'ACT' THEN 'active'
            WHEN raw_json->'ABN'->>'@status' = 'CAN' THEN 'deregistered'
            ELSE NULL
        END AS status

    FROM src_abn_bulk_extract
)
WHERE abn IS NOT NULL
""")

con.sql("""
SELECT *
FROM firmable_entities
LIMIT 20
""").show()

┌───────────┬─────────────┬───────────┬──────────────────────────────────────────┬────────────────────────────┬─────────┐
│ entity_id │     abn     │    acn    │                legal_name                │        entity_type         │ status  │
│   int64   │   varchar   │  varchar  │                 varchar                  │          varchar           │ varchar │
├───────────┼─────────────┼───────────┼──────────────────────────────────────────┼────────────────────────────┼─────────┤
│         1 │ 11000000948 │ 000000948 │ QBE INSURANCE (INTERNATIONAL) LTD        │ Australian Public Company  │ NULL    │
│         2 │ 11000002568 │ 000002568 │ TOOHEYS PTY LIMITED                      │ Australian Private Company │ NULL    │
│         3 │ 11000003314 │ 000003314 │ NEWCASTLE GOLF CLUB LTD                  │ Australian Public Company  │ NULL    │
│         4 │ 11000003378 │ 000003378 │ BBC HARDWARE LIMITED                     │ Australian Public Company  │ NULL    │
│         5 │ 1100000787

In [16]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT abn) AS unique_abn,
    COUNT(DISTINCT acn) AS unique_acn,
    COUNT(*) FILTER (WHERE abn IS NULL) AS null_abn,
    COUNT(*) FILTER (WHERE acn IS NULL) AS null_acn,
    COUNT(*) FILTER (WHERE legal_name IS NULL) AS null_legal_name
FROM firmable_entities
""").show()

┌─────────┬────────────┬────────────┬──────────┬──────────┬─────────────────┐
│  rows   │ unique_abn │ unique_acn │ null_abn │ null_acn │ null_legal_name │
│  int64  │   int64    │   int64    │  int64   │  int64   │      int64      │
├─────────┼────────────┼────────────┼──────────┼──────────┼─────────────────┤
│ 1027700 │    1027700 │     220719 │        0 │   806981 │          548120 │
└─────────┴────────────┴────────────┴──────────┴──────────┴─────────────────┘



In [17]:
con.sql("""
SELECT abn, COUNT(*) AS n
FROM firmable_entities
GROUP BY abn
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20
""").show()

┌─────────┬───────┐
│   abn   │   n   │
│ varchar │ int64 │
└─────────┴───────┘
      0 rows     



In [18]:
con.sql("""
WITH asic AS (
    SELECT
        TRIM(CAST("ACN" AS VARCHAR)) AS acn,
        CASE
            WHEN "ABN" IS NULL OR CAST("ABN" AS VARCHAR) IN ('0', '')
                THEN NULL
            ELSE TRIM(CAST("ABN" AS VARCHAR))
        END AS abn
    FROM src_company_dataset_current
)

SELECT
    COUNT(*) AS asic_rows,

    COUNT(*) FILTER (
        WHERE regexp_full_match(acn, '[0-9]{9}')
    ) AS valid_acn_rows,

    COUNT(*) FILTER (
        WHERE regexp_full_match(acn, '[0-9]{9}')
          AND NOT EXISTS (
              SELECT 1
              FROM firmable_entities f
              WHERE f.acn = asic.acn
          )
    ) AS new_entities_by_acn,

    COUNT(*) FILTER (
        WHERE abn IS NOT NULL
          AND EXISTS (
              SELECT 1
              FROM firmable_entities f
              WHERE f.abn = asic.abn
          )
    ) AS already_found_by_abn

FROM asic
""").show()

┌───────────┬────────────────┬─────────────────────┬──────────────────────┐
│ asic_rows │ valid_acn_rows │ new_entities_by_acn │ already_found_by_abn │
│   int64   │     int64      │        int64        │        int64         │
├───────────┼────────────────┼─────────────────────┼──────────────────────┤
│   4448915 │        4448915 │             4308397 │               140169 │
└───────────┴────────────────┴─────────────────────┴──────────────────────┘



In [19]:
con.execute("""
INSERT INTO firmable_entities
    (entity_id, abn, acn, legal_name, entity_type, status)

SELECT
    (SELECT COALESCE(MAX(entity_id), 0) FROM firmable_entities)
        + row_number() OVER (ORDER BY acn) AS entity_id,

    abn,
    acn,
    legal_name,
    'company' AS entity_type,
    status

FROM (
    SELECT
        CASE
            WHEN "ABN" IS NULL
              OR CAST("ABN" AS VARCHAR) IN ('0', '')
            THEN NULL
            ELSE TRIM(CAST("ABN" AS VARCHAR))
        END AS abn,

        TRIM(CAST("ACN" AS VARCHAR)) AS acn,

        TRIM("Company Name") AS legal_name,

        CASE
            WHEN "Status" = 'REGD' THEN 'active'
            WHEN "Status" = 'DRGD' THEN 'deregistered'
            ELSE NULL
        END AS status

    FROM src_company_dataset_current
) s

WHERE regexp_full_match(acn, '[0-9]{9}')

AND NOT EXISTS (
    SELECT 1
    FROM firmable_entities f
    WHERE f.acn = s.acn
)
""")

In [20]:
con.sql("""
SELECT
    COUNT(*) AS entities,
    COUNT(DISTINCT abn) AS abns,
    COUNT(DISTINCT acn) AS acns
FROM firmable_entities
""").show()

┌──────────┬─────────┬─────────┐
│ entities │  abns   │  acns   │
│  int64   │  int64  │  int64  │
├──────────┼─────────┼─────────┤
│  5336097 │ 3291068 │ 4115483 │
└──────────┴─────────┴─────────┘



In [21]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE abn IS NOT NULL) AS rows_with_abn,
    COUNT(DISTINCT abn) AS unique_abn,
    COUNT(*) FILTER (WHERE acn IS NOT NULL) AS rows_with_acn,
    COUNT(DISTINCT acn) AS unique_acn,

    COUNT(*) FILTER (WHERE abn IS NOT NULL)
        - COUNT(DISTINCT abn) AS duplicate_abn_rows,

    COUNT(*) FILTER (WHERE acn IS NOT NULL)
        - COUNT(DISTINCT acn) AS duplicate_acn_rows

FROM firmable_entities
""").show()

┌───────────────┬────────────┬───────────────┬────────────┬────────────────────┬────────────────────┐
│ rows_with_abn │ unique_abn │ rows_with_acn │ unique_acn │ duplicate_abn_rows │ duplicate_acn_rows │
│     int64     │   int64    │     int64     │   int64    │       int64        │       int64        │
├───────────────┼────────────┼───────────────┼────────────┼────────────────────┼────────────────────┤
│       3628432 │    3291068 │       4529116 │    4115483 │             337364 │             413633 │
└───────────────┴────────────┴───────────────┴────────────┴────────────────────┴────────────────────┘



In [22]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT "ACN") AS unique_acn,

    COUNT(*) FILTER (
        WHERE "Current Name Indicator" = 'Y'
    ) AS current_name_rows,

    COUNT(*) FILTER (
        WHERE "Current Name Indicator" IS NULL
    ) AS historical_name_rows

FROM src_company_dataset_current
""").show()

┌─────────┬────────────┬───────────────────┬──────────────────────┐
│  rows   │ unique_acn │ current_name_rows │ historical_name_rows │
│  int64  │   int64    │       int64       │        int64         │
├─────────┼────────────┼───────────────────┼──────────────────────┤
│ 4448915 │    4014773 │           4014734 │               434181 │
└─────────┴────────────┴───────────────────┴──────────────────────┘



In [23]:
con.sql("""
SELECT
    "ACN",
    "Company Name",
    "Current Name Indicator",
    "Current Name",
    "Current Name Start Date",
    "Status",
    "ABN"

FROM src_company_dataset_current

WHERE "ACN" IN (
    SELECT "ACN"
    FROM src_company_dataset_current
    GROUP BY "ACN"
    HAVING COUNT(*) > 1
    LIMIT 5
)

ORDER BY "ACN", "Current Name Start Date"
""").show(max_width=120)

┌───────────┬────────────────────────────────┬──────────────────────┬───┬──────────────────────┬─────────┬─────────────┐
│    ACN    │          Company Name          │ Current Name Indica… │ … │ Current Name Start … │ Status  │     ABN     │
│  varchar  │            varchar             │       varchar        │ … │         date         │ varchar │    int64    │
├───────────┼────────────────────────────────┼──────────────────────┼───┼──────────────────────┼─────────┼─────────────┤
│ 125557002 │ STRATUM FINANCIAL SERVICES PT… │ NULL                 │ … │ 2007-06-01           │ REGD    │ 89125557002 │
│ 125557002 │ MERIT WEALTH PTY LTD           │ Y                    │ … │ NULL                 │ REGD    │ 89125557002 │
│ 125566396 │ CAMPAIGN KITCHENS PTY. LTD.    │ NULL                 │ … │ 2007-06-13           │ REGD    │           0 │
│ 125566396 │ CAMPAIGN KITCHEN PTY LTD       │ Y                    │ … │ NULL                 │ REGD    │           0 │
│ 125653992 │ GLO CONSTRUCTIONS 

In [24]:
from pathlib import Path
import pandas as pd

CANONICAL_DIR = PROJECT_ROOT / "outputs" / "part2" / "canonical"

dataset_id = "4117fdec-3eff-44a1-9b78-869f7ec5c409"

canonical_path = (
    CANONICAL_DIR
    / dataset_id
    / "canonical.records.jsonl"
)

df_canonical = pd.read_json(
    canonical_path,
    lines=True
)

print("Rows:", len(df_canonical))
print("Columns:")
print(df_canonical.columns.tolist())

display(df_canonical.head(10))

Rows: 1000
Columns:
['entity', 'address', 'observation', '_meta']


,entity,address,observation,_meta
0,"{'legal_name': 'A D DANIELI AUDIT PTY LTD', 'a...","{'locality': 'SYDNEY', 'state': 'NSW', 'postco...",{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'entity.legal_name': 0.9...
1,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
2,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
3,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
4,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
5,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
6,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
7,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
8,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
9,{},{},{'observed_at': '2026-09-27T23:02:36.782719+00...,{'field_confidence': {'observation.source_reco...
